# 전처리 & 머신러닝 통합 개인과제

## Part 1. 고객 데이터 품질 개선

### 실무 시나리오

전자상거래 기업의 CRM팀은 고객별 구매 패턴을 분석하고 구매 가능성이 높은 **고객을 선별**하려고 합니다.  
그러나 전달받은 원본 고객 데이터에는 잘못된 자료형, 범주 표기 불일치, 결측치, 중복 데이터와 극단값이 포함되어 있어 바로 분석에 사용할 수 없습니다.

CRM팀은 **데이터 분석 담당자**로서 원본 데이터의 품질 문제를 진단하고, 이후 Part 2의 EDA와 Part 3의 구매 예측 모델링에 사용할 수 있는 분석용 데이터셋을 만들어야 합니다.

### 과제 목표

- 데이터의 구조·자료형·기초 분포를 확인하고 주요 품질 문제를 파악합니다.
- 결측치, 중복값, 이상치와 범주 표기 불일치를 분석 목적에 맞게 처리합니다.
- 조건 기반 데이터 추출과 파생변수 생성을 수행합니다.
- 전처리 결과를 검증하고 다음 Part에서 사용할 CSV 파일로 저장합니다.

### 사용 환경 및 데이터

- Python 3.X
- pandas, numpy
- `전자상거래_고객구매_원본데이터.csv`

### 제출 결과물

- Part 1 실습 노트북
- `전자상거래_고객구매_전처리완료.csv`

> 문제 1~3은 필수 문제입니다.

## 문제 1. 원본 고객 데이터 품질 진단

### 업무 상황

CRM팀에 분석 일정을 공유하기 전에 원본 데이터가 실제 분석에 사용할 수 있는 상태인지 확인해야 합니다.  
데이터를 수정하기 전에 구조와 품질 문제를 먼저 점검하고, 이후 처리해야 할 항목을 정리하세요.

### 요구사항

1. 원본 데이터를 불러오고 데이터의 크기, 컬럼, 자료형과 기초 통계량을 확인하세요.
2. 컬럼별 결측치와 전체 행 기준 중복 데이터를 확인하세요.
3. 주요 범주형 컬럼의 값을 확인하여 표기 불일치 여부를 파악하세요.
4. 이후 정제가 필요한 주요 품질 문제를 간단히 정리하세요.

### 힌트

- 데이터 구조와 기초 통계량을 함께 확인하면 자료형 오류와 비정상 범위를 찾기 쉽습니다.
- 범주형 컬럼은 고유값을 확인하여 대소문자, 공백, 한글·영문 혼용 여부를 살펴볼 수 있습니다.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("전자상거래_고객구매_데이터.csv")

# TODO: 원본 데이터를 불러와 df에 저장하세요.
df = pd.read_csv(DATA_PATH)

# TODO: 데이터의 크기, 컬럼, 자료형과 기초 통계량을 확인하세요.
print("데이터 크기:", df.shape)
print("[컬럼]")
display(df.dtypes)
display("[기초 통계량]")
print(df.describe())

# TODO: 결측치, 중복 데이터와 주요 범주형 컬럼의 값을 확인하세요.
print("[결측치]")
print(df.isnull().sum())
print("중복 데이터 개수:", df.duplicated().sum())
print("범주형 컬럼:",df.value_counts())

df.head()

데이터 크기: (1560, 15)
[컬럼]


CustomerID                   str
Gender                       str
Age                          str
Region                       str
MembershipLevel              str
VisitCount                 int64
AveragePurchaseAmount        str
TotalPurchaseAmount      float64
SatisfactionScore        float64
LastPurchaseDate             str
CouponUsed                   str
PreferredCategory            str
SignupDate                   str
Email                        str
PurchaseStatus             int64
dtype: object

'[기초 통계량]'

        VisitCount  TotalPurchaseAmount  SatisfactionScore  PurchaseStatus
count  1560.000000         1.560000e+03        1521.000000     1560.000000
mean      9.521795         4.841402e+05           3.662853        0.485256
std       8.146008         1.410879e+06           0.759151        0.499943
min       1.000000         5.000000e+03           1.100000        0.000000
25%       7.000000         1.869500e+05           3.200000        0.000000
50%       9.000000         3.366000e+05           3.700000        0.000000
75%      11.000000         5.370750e+05           4.200000        1.000000
max     170.000000         3.000000e+07           5.000000        1.000000
[결측치]
CustomerID                0
Gender                   26
Age                      42
Region                   31
MembershipLevel          22
VisitCount                0
AveragePurchaseAmount    45
TotalPurchaseAmount       0
SatisfactionScore        39
LastPurchaseDate         30
CouponUsed                0
PreferredCa

,CustomerID,Gender,Age,Region,MembershipLevel,VisitCount,AveragePurchaseAmount,TotalPurchaseAmount,SatisfactionScore,LastPurchaseDate,CouponUsed,PreferredCategory,SignupDate,Email,PurchaseStatus
0,CUST101486,Female,40,NaN,silver,5,75000.0,139800.0,3.7,2026-02-07,N,Beauty,2022-12-03,customer1486@example.com,0
1,CUST100113,Male,39,GYEONGGI,Basic,7,74400.0,159200.0,4.8,2026-04-22,N,Home,2021-07-16,customer113@example.com,1
2,CUST100168,Male,105,Gwangju,Silver,12,185800.0,1055800.0,4.5,2026-06-06,N,Beauty,2022-02-15,customer168@example.com,1
3,CUST101212,Male,44,Daegu,VIP,7,153900.0,413200.0,4.5,2026-05-07,N,Home,2021-10-27,customer1212@example.com,1
4,CUST100312,Male,37,Gyeonggi,Basic,14,123700.0,576000.0,4.0,2026-04-16,n,Home,2025-11-08,customer312@example.com,1


### 품질 진단 결과

- 자료형 정리가 필요한 컬럼: Age, AveragePurchaseAmount, LastPurchaseDate, SignupDate
- 표기 통일이 필요한 컬럼: Gender, Region, MembershipLevel, CouponUsed, PreferredCategory
- 결측치가 있는 컬럼: AveragePurchaseAmount, Age, SatisfactionScore, Region,LastPurchaseDate, Gender, PreferredCategory, MembershipLevel
- 중복 데이터 확인 결과: 60
- 이상치 확인이 필요한 컬럼: Age, VisitCount, TotalPurchaseAmount
- 이후 처리할 주요 항목
  1) 자료형 정리
  2) 컬럼 표기 통일
  3) 중복행 60건 제거
  4) 결측치 처리
  5) 이상치 처리

## 문제 2. 분석 가능한 고객 데이터로 정제

### 업무 상황

품질 진단 결과를 바탕으로 고객 데이터를 분석 가능한 상태로 정리해야 합니다.  
처리 과정에서 고객의 실제 구매 행동 정보가 불필요하게 손실되지 않도록 데이터 특성을 고려하세요.

### 요구사항

1. 원본 데이터를 보존한 상태에서 정제용 데이터를 생성하세요.
2. 분석에 맞지 않는 수치형·날짜형 자료형과 범주 표기를 정리하세요.
3. 결측치와 전체 행 기준 중복 데이터를 처리하세요.
4. 주요 수치형 컬럼의 이상치를 탐지하고 적절한 방법으로 처리하세요.
5. 처리 전후의 결측치, 중복값과 이상치 상태를 확인하세요.

### 힌트

- 변환할 수 없는 문자열은 결측치로 바꾼 뒤 일관되게 처리할 수 있습니다.
- 이상치는 무조건 삭제하기보다 실제 우수 고객의 행동일 가능성도 고려하세요.
- IQR은 이상치 후보를 확인하는 대표적인 방법입니다.

In [26]:
# TODO: 원본을 보존하고 정제용 데이터 df_clean을 생성하세요.
df_clean = df.copy()

# TODO: 수치형·날짜형 자료형과 범주 표기를 정리하세요.

# [자료형 정리]
# [Age] 변환 (문자 → 숫자)
if pd.api.types.is_string_dtype(df_clean['Age']):
    df_clean['Age'] = df_clean['Age'].replace({'thirty': '30'})        # 영어-> 숫자 문자열
    df_clean['Age'] = df_clean['Age'].str.extract(r'(\d+)')[0]         # 숫자만 추출 (글자,띄어쓰기 제거)
df_clean['Age'] = pd.to_numeric(df_clean['Age'], errors='coerce')  # '?','미입력','-','none','unknown'-> NaN, 숫자형으로 변환
#df_clean['Age'].unique() # -------- 이상치 130 변경해야하는 거 남았음 / NaN 은 처리해야함

# [AveragePurchaseAmount] 변환 (문자 → 숫자)
if pd.api.types.is_string_dtype(df_clean['AveragePurchaseAmount']):
    df_clean['AveragePurchaseAmount'] = df_clean['AveragePurchaseAmount'].str.replace(',', '', regex=False) # 콤마 제거
df_clean['AveragePurchaseAmount'] = pd.to_numeric(df_clean['AveragePurchaseAmount'], errors='coerce') # 결측치 -> NaN, 숫자형으로 변환
# df_clean['AveragePurchaseAmount'].unique()
# AveragePurchaseAmount NaN 처리해야함

# ['LastPurchaseDate'] (문자 → 날짜)
if pd.api.types.is_string_dtype(df_clean['LastPurchaseDate']):
    df_clean['LastPurchaseDate'] = df_clean['LastPurchaseDate'].str.replace('.', '-', regex=False).str.replace('/', '-', regex=False) # '.,/' '-'로 통일
df_clean['LastPurchaseDate'] = pd.to_datetime(df_clean['LastPurchaseDate'], errors='coerce') #결측치 -> NaT, 날짜형으로 변환
# df_clean['LastPurchaseDate'].unique()
# Nat 처리해야함

# SignupDate (문자 → 날짜)
date_ymd = pd.to_datetime(df_clean['SignupDate'], format='%Y-%m-%d', errors='coerce') #연-월-일
date_dmy = pd.to_datetime(df_clean['SignupDate'], format='%d-%m-%Y', errors='coerce') #일-월-연
df_clean['SignupDate'] = date_ymd.fillna(date_dmy) # 합치기
# 결측치 X / 회원가입일 보다 구매일이 더 빠른 거 있음 처리 필요
# df_clean['SignupDate'].unique()


#=============================================================================================================
# [표기 통일]
df_clean['Gender'] = df_clean['Gender'].str.strip().str.title().replace({'F': 'Female', 'M': 'Male'}) 
#df_clean['Gender'].unique()

df_clean['Region'] = df_clean['Region'].str.strip().str.title() # 띄어쓰기 제거
# df_clean['Region'].unique()

df_clean['MembershipLevel'] = df_clean['MembershipLevel'].str.strip().str.title().replace({'Vip': 'VIP'})
# df_clean['MembershipLevel'].unique()

df_clean['CouponUsed'] = df_clean['CouponUsed'].str.strip().str.upper().replace({'YES': 'Y', 'NO': 'N'})
# df_clean['CouponUsed'].unique()

df_clean['PreferredCategory'] = df_clean['PreferredCategory'].str.strip().str.title()
# df_clean['PreferredCategory'].unique()


In [ ]:
# TODO: 중복 데이터를 처리하세요.
# [중복 데이터 처리]
print('중복 데이터 제거 전 행 개수:', len(df_clean))
print('중복행 개수:', df_clean.duplicated().sum())

df_clean = df_clean.drop_duplicates()

# TODO: 처리 전후 결과를 확인하세요.
print('제거 후 행 개수:', len(df_clean))

제거 전 행 개수: 1560
중복행 개수: 60
제거 후 행 개수: 1500


In [42]:
# TODO: 결측치를 처리하세요.
print('결측치 제거 전 행 개수:', len(df_clean))
print('[결측치 개수]', df_clean.isnull().sum())

df_clean.dropna(axis=0, inplace=True) # 결측치 처리

# TODO: 처리 전후 결과를 확인하세요.

print('제거 후 행 개수:', len(df_clean))

df_clean['Age'].unique()

결측치 제거 전 행 개수: 1259
[결측치 개수] CustomerID               0
Gender                   0
Age                      0
Region                   0
MembershipLevel          0
VisitCount               0
AveragePurchaseAmount    0
TotalPurchaseAmount      0
SatisfactionScore        0
LastPurchaseDate         0
CouponUsed               0
PreferredCategory        0
SignupDate               0
Email                    0
PurchaseStatus           0
dtype: int64
제거 후 행 개수: 1259


array([ 39., 105.,  44.,  37.,  47.,  26.,  56.,  42.,  53.,  23.,  34.,
        46.,  52.,  31.,  29.,  43.,  50.,  18.,  45.,  25.,  57.,  20.,
        54.,  22.,  51.,  21.,  41.,  28.,  38.,  49.,  27.,  40.,  62.,
        24.,  30.,  36.,  33.,  63.,  32.,  60.,  48.,  61.,  35.,  58.,
        55.,  59.,  19.,  72.,  79., 120.,  76.,  65.,  66., 112.,  70.,
        69., 130.,  99., 101.,  68.,  67.,  64.])

In [ ]:
    # TODO: Q1, Q3, IQR, 하한과 상한을 반환하세요.

# Q1, Q3, IQR, lower, upper 함수
def get_iqr_bounds(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return Q1, Q3, IQR, lower_bound, upper_bound

# 컬럼 넣기
cols = ['Age', 'VisitCount', 'AveragePurchaseAmount', 'TotalPurchaseAmount', 'SatisfactionScore']
bounds = {}
for col in cols:
    Q1, Q3, IQR, lower_bound, upper_bound = get_iqr_bounds(df_clean[col])
    bounds[col] = {'Q1': Q1, 'Q3': Q3, 'IQR': IQR, 'lower': lower_bound, 'upper': upper_bound}

# 반환
print(f"Age",bounds['Age'])
print(f"VisitCount",bounds['VisitCount'])
print(f"AveragePurchaseAmount",bounds['AveragePurchaseAmount'])
print(f"TotalPurchaseAmount",bounds['TotalPurchaseAmount'])
print(f"SatisfactionScore",bounds['SatisfactionScore'])


# TODO: 주요 수치형 컬럼의 이상치를 탐지하고 처리하세요.
# Age / VisitCount / AveragePurchaseAmount / TotalPurchaseAmount / SatisfactionScore
def get_outliers(df_clean, col, bounds):
    lower_bound = bounds[col]['lower']
    upper_bound = bounds[col]['upper']
    outliers = df_clean[(df_clean[col] < lower_bound) | (df_clean[col] > upper_bound)]
    return outliers

#모델링용 이상치 제거 결과
df_removed = df_clean.copy()
for col in cols:
    lower_bound = bounds[col]['lower']
    upper_bound = bounds[col]['upper']
    df_removed = df_removed[(df_removed[col] >= lower_bound) & (df_removed[col] <= upper_bound)]

# TODO: 처리 전후 이상치 상태를 확인하세요

print("처리 전 행 개수:", len(df_clean))
print("처리 후 행 개수:", len(df_removed))


Age {'Q1': np.float64(32.0), 'Q3': np.float64(47.0), 'IQR': np.float64(15.0), 'lower': np.float64(9.5), 'upper': np.float64(69.5)}
VisitCount {'Q1': np.float64(7.0), 'Q3': np.float64(11.0), 'IQR': np.float64(4.0), 'lower': np.float64(1.0), 'upper': np.float64(17.0)}
AveragePurchaseAmount {'Q1': np.float64(83650.0), 'Q3': np.float64(152100.0), 'IQR': np.float64(68450.0), 'lower': np.float64(-19025.0), 'upper': np.float64(254775.0)}
TotalPurchaseAmount {'Q1': np.float64(186350.0), 'Q3': np.float64(542200.0), 'IQR': np.float64(355850.0), 'lower': np.float64(-347425.0), 'upper': np.float64(1075975.0)}
SatisfactionScore {'Q1': np.float64(3.1), 'Q3': np.float64(4.2), 'IQR': np.float64(1.1), 'lower': np.float64(1.45), 'upper': np.float64(5.8500000000000005)}
처리 전 행 개수: 1259
처리 후 행 개수: 1183


## 문제 3. 분석용 변수 생성 및 다음 단계 데이터 준비

### 업무 상황

정제된 데이터를 CRM팀의 고객군 분석과 구매 예측 업무에 활용하려면, 원본 컬럼만으로 확인하기 어려운 고객 특성을 분석용 변수로 표현해야 합니다.  
필요한 고객을 조건에 따라 추출하고, 이후 Part 2와 Part 3에서 활용할 파생변수를 만든 뒤 최종 데이터를 저장하세요.

### 요구사항

1. 정제된 데이터에서 업무적으로 의미 있는 조건을 설정하여 고객 데이터를 추출하세요.
2. 고객 분석에 활용할 파생변수를 **2개 이상** 생성하세요.
3. 생성한 파생변수의 값과 분포가 적절한지 확인하세요.
4. 최종 데이터의 결측치, 중복값, 고객 식별자와 타깃 값을 검증하세요.
5. 전처리 완료 데이터를 `전자상거래_고객구매_전처리완료.csv`로 저장하세요.

### 힌트

- 연령대, 구매 수준, 최근 구매 여부, 우수 고객 여부 등을 파생변수 후보로 고려할 수 있습니다.
- 조건 추출 결과는 별도 DataFrame으로 확인해도 되며, 최종 데이터 전체를 삭제할 필요는 없습니다.

- 고객 연령을 구간화한 `AgeGroup`을 생성하세요.
- 총구매금액을 기준으로 `PurchaseGrade`를 생성하세요.

In [76]:
REFERENCE_DATE = pd.Timestamp("2026-06-01")

# TODO: 업무적으로 의미 있는 조건을 설정하여 고객 데이터를 추출하세요.
# AgeGroup 고객 데이터 추출
def age_group(age):
    if age <= 19:
        return '10대'
    elif age <= 29:
        return '20대'
    elif age <= 39:
        return '30대'
    elif age <= 49:
        return '40대'
    elif age <= 59:
        return '50대'
    else:
        return '60대 이상'


# totalpurchaseamount 데이터 추출
q1 = df_clean['TotalPurchaseAmount'].quantile(0.25)
q2 = df_clean['TotalPurchaseAmount'].quantile(0.50) 
q3 = df_clean['TotalPurchaseAmount'].quantile(0.75)

def PurchaseGrade(amount):
    if amount <= q1:
        return '25% 미만'
    elif amount < q2:
        return '25~50% 미만'
    elif amount < q3:
        return '50~75% 미만'
    else:
        return '75% 이상'
    


# TODO: 분석용 파생변수를 2개 이상 생성하세요.
# AgeGroup, PurchaseGrade
# AgeGroup 파생변수 생성
df_clean['AgeGroup'] = df_clean['Age'].apply(age_group)
df_clean['PurchaseGrade'] = df_clean['TotalPurchaseAmount'].apply(PurchaseGrade)


# TODO: 파생변수의 값과 분포를 확인하세요.
df_clean['AgeGroup'].value_counts()
df_clean['PurchaseGrade'].value_counts()

df_clean.describe()

,Age,VisitCount,AveragePurchaseAmount,TotalPurchaseAmount,SatisfactionScore,LastPurchaseDate,SignupDate,PurchaseStatus
count,1259.000000,1259.000000,1259.000000,1.259000e+03,1259.000000,1259,1259,1259.000000
mean,39.497220,9.460683,120874.583002,4.772330e+05,3.654885,2026-04-14 13:11:29.118348,2023-03-26 16:30:30.023828,0.490071
min,18.000000,2.000000,5000.000000,5.300000e+03,1.100000,2025-08-15 00:00:00,2020-06-26 00:00:00,0.000000
25%,32.000000,7.000000,83650.000000,1.863500e+05,3.100000,2026-03-19 00:00:00,2021-10-27 12:00:00,0.000000
50%,39.000000,9.000000,117500.000000,3.374000e+05,3.700000,2026-04-24 00:00:00,2023-04-16 00:00:00,0.000000
75%,47.000000,11.000000,152100.000000,5.422000e+05,4.200000,2026-05-23 00:00:00,2024-08-05 00:00:00,1.000000
max,130.000000,170.000000,280400.000000,3.000000e+07,5.000000,2026-06-29 00:00:00,2026-01-02 00:00:00,1.000000
std,11.850578,7.652987,52533.952430,1.366631e+06,0.761399,NaN,NaN,0.500100


In [86]:
OUTPUT_PATH = Path("전자상거래_고객구매_전처리완료.csv")

# TODO: 최종 데이터 품질을 검증하세요.

df_clean.isna().sum()
missing_count = df_clean.isna().sum().sum()
print("전체 결측 수 (0이어야 함):", missing_count)
duplicate_count = df_clean.duplicated().sum()
print("전체 행 중복 수 (0이어야 함):", duplicate_count)
id_duplicate_count = df_clean["CustomerID"].duplicated().sum()
id_missing_count = df_clean["CustomerID"].isna().sum()
print("ID 중복 수:", id_duplicate_count)
print("ID 결측 수:", id_missing_count)
id_text = df_clean["CustomerID"].astype("string")
id_text = id_text.str.strip()
id_empty_count = (id_text == "").sum()
print("공백만 있는 ID 수 (0이어야 함):", id_empty_count)

df_clean["PurchaseStatus"].value_counts(dropna=False)
target_valid = df_clean["PurchaseStatus"].isin([0, 1]).all()
print("타깃이 모두 0 또는 1인가 (True여야 함):", target_valid)
print("AgeGroup" in df_clean.columns)
print("PurchaseGrade" in df_clean.columns)

assert missing_count == 0, "결측치가 남아 있습니다."
assert duplicate_count == 0, "전체 행 중복이 남아 있습니다."
assert id_duplicate_count == 0, "고객 ID가 중복됩니다."
assert id_missing_count == 0, "고객 ID가 비어 있습니다."
assert id_empty_count == 0, "공백만 있는 고객 ID가 있습니다."
assert target_valid, "타깃에 0/1 이외의 값이 있습니다."
assert "AgeGroup" in df_clean.columns
assert "PurchaseGrade" in df_clean.columns
print("최종 품질 검증 통과:", df_clean.shape)


# TODO: 전처리 완료 데이터를 CSV 파일로 저장하세요.
df_to_save = df_clean.copy()
df_to_save["LastPurchaseDate"] = df_to_save["LastPurchaseDate"].dt.strftime("%Y-%m-%d")
df_to_save["SignupDate"] = df_to_save["SignupDate"].dt.strftime("%Y-%m-%d")
df_to_save.to_csv("전자상거래_고객구매_전처리완료.csv", index=False, encoding="utf-8-sig")
saved_df = pd.read_csv("전자상거래_고객구매_전처리완료.csv")
saved_df.head()
print("정제 데이터 크기:", df_clean.shape)
print("다시 읽은 크기:", saved_df.shape)
print("컬럼 일치:", saved_df.columns.equals(df_clean.columns))
saved_df.isna().sum()
assert saved_df.shape == df_clean.shape
assert saved_df.columns.equals(df_clean.columns)
assert saved_df.isna().sum().sum() == 0
assert saved_df["CustomerID"].is_unique
print("CSV 저장 및 다시 읽기 확인 완료")


전체 결측 수 (0이어야 함): 0
전체 행 중복 수 (0이어야 함): 0
ID 중복 수: 0
ID 결측 수: 0
공백만 있는 ID 수 (0이어야 함): 0
타깃이 모두 0 또는 1인가 (True여야 함): True
True
True
최종 품질 검증 통과: (1259, 17)
정제 데이터 크기: (1259, 17)
다시 읽은 크기: (1259, 17)
컬럼 일치: True
CSV 저장 및 다시 읽기 확인 완료


# 문제 4. 데이터 정제 및 파생변수 설계 근거 설명

## 업무 상황

CRM팀은 전처리 결과가 단순히 실행되는 것뿐 아니라,
적용한 처리 기준과 파생변수가 실제 분석 목적에 적합한지 확인하려고 합니다.

문제 1~3에서 수행한 결과를 바탕으로 다음 내용을 설명하세요.

## 요구사항

### 4-1. 데이터 품질 문제의 영향과 한계

1. 문제 1에서 확인한 주요 품질 문제를 한 가지 이상 선택하세요. 
-> 표기가 다 다름
2. 해당 문제가 이후 EDA 또는 모델링 결과에 미칠 수 있는 영향을 설명하세요. 
-> 제대로 매핑이 안 될 가능성이 있음
3. 현재 데이터만으로 판단하기 어려운 점이나 추가 확인이 필요한 조건을 작성하세요. -> 가입일자보다 구매일자가 더 빠른 데이터가 있어 추가 확인이 필요


### 4-2. 결측치·이상치 처리 방법의 선택 근거

1. 적용한 결측치 처리 방법을 한 가지 이상 제시하세요.
-> 제거
2. 해당 방법을 선택한 이유를 데이터 특성과 연결하여 작성하세요.
-> 중앙값이나 평균값 등으로 함부로 채울 수 없는 값들이었으므로 제거함
3. 적용한 이상치 처리 방법과 처리 시 주의할 점을 설명하세요.
-> IQR

### 4-3. 파생변수의 활용 의도와 한계

1. `파생변수1`과 `파생변수2`의 생성 기준을 설명하세요.
-> 고객 연령을 구간화한 `AgeGroup`과 총구매금액을 기준으로 `PurchaseGrade`을 생성함
2. 각 파생변수가 Part 2 고객 특성 분석에서 어떻게 활용될 수 있는지 작성하세요. -> 고객연령대 별로, 금액대 별로 고객 특성을 파악해볼 수 있음
3. 파생변수 사용 시 발생할 수 있는 정보 손실 또는 해석상의 한계를 설명하세요. -> 파생변수를 쓰게 되면 정확한 연령대 및 금액 추정이 불가함

## 문제 4 결과 작성

### 4-1. 데이터 품질 문제의 영향과 한계

- 선택한 품질 문제: LastPurchaseDate에서 '.', '/' 이 섞여있어 저장되어있었으며, 년-월-일과 월-일-년이 섞여 저장되어있었다.
- EDA 또는 모델링에 미치는 영향: 통일되어있지 않아 날짜 파악에 어려움이 있음
- 추가 확인이 필요한 조건 또는 한계: 이를 그대로 사용하여 날짜가 달라지기 때문에 통일이 필요함

### 4-2. 결측치·이상치 처리 방법의 선택 근거

- 결측치 처리 방법: 제거
- 해당 방법을 선택한 이유: 나이, 금액 등의 카테고리는 함부로 채울 경우 왜곡이 발생할 수 있음
- 이상치 처리 방법: IQR
- 이상치 처리 시 주의점: 이상치가 아님에도 불구하고 삭제될 가능성이 있음

### 4-3. 파생변수의 활용 의도와 한계

- `AgeGroup` 생성 기준과 활용 목적: 연령대별로 고객 특성 파악
- `PurchaseGrade` 생성 기준과 활용 목적: 금액대 별로 고객 특성파악
- 파생변수 사용 시 한계: 파생변수를 쓰게되면 정확한 연령대 및 금액 추정이 불가함